# Scaling Laws: 모델 크기의 법칙 - 실습 코드 2: 실제 모델로 Scaling Law 검증 실험

- Tutorial ID: `expand-scaling-laws`
- Tutorial: Scaling Laws: 모델 크기의 법칙
- Section ID: `expand-scaling-laws-code-2`
- Section: 실습 코드 2: 실제 모델로 Scaling Law 검증 실험

이 노트북은 크기가 다른 여러 개의 작은 Transformer 모델을 직접 학습시켜보면서, "모델이 커지면 성능(Loss)이 일정한 규칙을 따라 좋아진다"는 **Scaling Law(스케일링 법칙)** 가 실제로 나타나는지 눈으로 확인해보는 실습입니다.

처음 이 주제를 공부하시는 분들도 따라올 수 있도록,
- 새로운 개념은 코드에 등장하기 **직전에** 먼저 말로 설명하고,
- 작은 숫자로 된 **장난감 예제**를 먼저 풀어본 뒤에 실제 실험으로 넘어가고,
- 실험이 실패하는 경우(함정)도 일부러 보여드리면서 "왜 실패하는지"까지 함께 이해하도록 구성했습니다.


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: 실제 모델로 Scaling Law 검증 실험
#
# 이 노트북은 "정답 코드를 한 번 실행"하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적한다
#   2) Power Law(거듭제곱 법칙)를 로그-로그 그래프와 선형회귀로 "검증"하는 방법을 이해한다
#   3) Causal Mask(인과적 마스크)가 왜 필요한지, 없으면 어떤 문제가 생기는지 직접 실험으로 확인한다
#   4) 학습 데이터에 "배울 거리(패턴)"가 있는지 없는지가 결과에 어떤 영향을 주는지 직접 실험으로 확인한다
#   5) 크기가 다른 여러 개의 실제 Transformer 모델을 학습시켜, 모델 크기와 최종 Loss 사이의 관계를 측정한다
#
# 전체 구성 (위에서 아래 순서로):
#   0. 라이브러리 불러오기
#   1. 복습 — Power Law란 무엇이고, 왜 로그-로그 그래프로 확인하는가 (가짜 데이터로 몸풀기)
#   2. 실험에서 사용할 용어(하이퍼파라미터) 미리 정리하기
#   3. 미니 실습 — 아주 작은 숫자로 shape의 흐름 따라가기
#   4. Causal Mask 개념 설명 + 실제로 마스크가 어떻게 생겼는지 확인
#   5. TinyTransformer 모델 클래스 정의 (미니 실습에서 본 내용을 그대로 클래스로 정리)
#   6. 실험에서 흔히 빠지는 함정 두 가지를 직접 재현해보기
#        (함정 1) Causal Mask가 없으면?
#        (함정 2) 데이터에 아무 패턴이 없으면?
#   7. 진짜 실험 — 5가지 크기의 모델을 학습시켜 Scaling Law 검증
#   8. 결과 표 정리 + Power Law 피팅 + 그래프로 확인
#   9. 결과 해석과 한계, 더 나아가기
#
# 읽는 순서 (각 코드 셀 안에서):
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰 데이터가 어떻게 만들어지는지 봅니다.
#   3) 가중치 행렬(임베딩, Linear 등)이 어떤 공간으로 값을 옮기는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, 스텝 수, 학습률 등을 바꿔 결과가 어떻게 변하는지 스스로 실험해봅니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape 변화"와 "정보가 이동하는 방향"을 보세요.
#   - 이 노트북의 실험은 강의/논문 재현이 아니라 "작은 규모로 같은 원리를 체험"하는 것이 목적입니다.
#     따라서 여기서 얻는 alpha 값은 논문의 값(0.076)과 다르게 나오는 것이 정상입니다. (9장에서 이유를 설명합니다.)
#   - torch 의존 코드는 Colab/로컬/서버 등 GPU가 없어도 실행되지만, CPU에서는 전체 실행에 몇 분 정도 걸릴 수 있습니다.
#     GPU가 있다면 자동으로 GPU를 사용하도록 코드를 작성해두었습니다.

## 0. 준비: 라이브러리 불러오기

먼저 이 노트북 전체에서 사용할 도구들을 불러옵니다.

| 라이브러리 | 용도 |
|---|---|
| `torch`, `torch.nn` | Transformer 모델을 만들고 학습시키는 데 사용하는 핵심 딥러닝 라이브러리 |
| `numpy` | 로그를 취하거나, 직선을 피팅(회귀)하는 등 숫자 계산에 사용 |
| `matplotlib` | 학습 곡선, Power Law 그래프 등을 그리는 데 사용 |
| `koreanize_matplotlib` | matplotlib 그래프에 한글이 깨지지 않고 잘 보이도록 폰트를 자동으로 설정해주는 작은 패키지 |
| `time` | 각 모델을 학습시키는 데 걸리는 시간을 재기 위해 사용 |
| `math` | 로그값(`math.log`) 등 기본적인 수학 함수를 사용하기 위해 사용 |

> 참고: 원본 코드에는 `DataLoader`, `Dataset`도 import 되어 있었지만, 이번 실습에서는 데이터가 아주 단순해서 배치를 직접 텐서 슬라이싱으로 만듭니다. 실제 프로젝트에서 대용량 데이터를 다룰 때는 `DataLoader`/`Dataset`을 사용하는 것이 일반적입니다.

In [ ]:
# koreanize_matplotlib이 아직 설치되어 있지 않다면 설치합니다 (그래프의 한글 깨짐 방지용).
# 이미 설치되어 있다면 아무 일도 하지 않고 빠르게 지나갑니다.
%pip install -q koreanize-matplotlib

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import math

try:
    import koreanize_matplotlib  # import만 해도 matplotlib의 기본 폰트가 한글 지원 폰트로 바뀝니다
except ImportError:
    # 혹시 설치가 잘 안 되었더라도 여기서 노트북 전체가 멈추지 않도록 방어적으로 처리합니다.
    # (이 경우 그래프의 한글 글자가 네모(□)로 깨져 보일 수 있지만, 실습 자체에는 지장이 없습니다.)
    print("koreanize_matplotlib을 불러오지 못했습니다. 그래프의 한글이 깨져 보일 수 있습니다.")

# GPU(cuda)가 있으면 GPU를, 없으면 CPU를 자동으로 사용하도록 설정합니다.
# 이렇게 device를 변수로 만들어두고 .to(device)를 붙여주면,
# 같은 코드가 GPU가 있는 환경(Colab 등)과 없는 환경(로컬 CPU) 모두에서 잘 동작합니다.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용할 device: {device}")

## 1. 복습: Power Law(거듭제곱 법칙)란 무엇인가?

본격적인 실험에 들어가기 전에, "Scaling Law를 어떻게 숫자로 확인하는지"부터 가짜(합성) 데이터로 가볍게 몸풀기를 해보겠습니다. 이미 이전 실습(실습 코드 1)에서 다뤘다면 가볍게 복습하는 셈 치고 넘어가시면 됩니다.

**Scaling Law**는 보통 다음과 같은 **Power Law(거듭제곱 법칙)** 형태로 나타납니다.

$$L(N) = C \cdot N^{-\alpha}$$

- $N$ : 모델 크기(파라미터 수)
- $L(N)$ : 모델 크기가 $N$일 때의 Loss (작을수록 좋은 모델)
- $\alpha$ (alpha) : 모델을 키울수록 Loss가 얼마나 빨리 줄어드는지를 나타내는 지수. 이 값이 클수록 "모델을 키우는 효과"가 큽니다.
- $C$ : 상수 (그래프의 전체적인 높이를 결정)

문제는, 이 식은 $N$과 $L$이 곱셈·거듭제곱으로 얽혀 있어서 눈으로 보고 "아, $\alpha$가 얼마구나"를 바로 알아채기 어렵다는 점입니다. 여기서 로그(log)를 이용한 트릭이 등장합니다.

### 로그를 취하면 왜 직선이 될까?

양변에 로그를 취해보겠습니다.

$$\log L(N) = \log C - \alpha \log N$$

이 식을 $y = \log L$, $x = \log N$이라고 이름 붙이면,

$$y = (\log C) + (-\alpha) \cdot x$$

이 되어, 정확히 **직선의 방정식** $y = b + m x$ 꼴이 됩니다! 여기서
- 기울기 $m = -\alpha$
- 절편 $b = \log C$

즉, **$\log N$ 대 $\log L$을 그래프로 그리면 직선이 되어야 하고, 그 직선의 기울기에 마이너스 부호를 붙이면 바로 $\alpha$**가 됩니다. 이것이 바로 아래에서 사용할 `np.polyfit(log_N, log_L, 1)` (1차 직선으로 피팅)의 원리입니다.

참고로 Kaplan et al. (2020) 논문에서는 모델 파라미터 수에 대한 지수를 $\alpha_N \approx 0.076$으로 보고했습니다. (이 값은 특정 데이터셋·토크나이저·파라미터 집계 방식에서 얻어진 값이라, 다른 설정에서는 상수가 달라질 수 있다는 점도 함께 기억해두면 좋습니다 — 자세한 내용은 9장에서 다시 다룹니다.)

### 먼저 "정답을 알고 있는" 가짜 데이터로 연습해보기

아래 코드에서는 우리가 **직접 정한 $\alpha$ 값(0.15)** 으로 가짜 데이터를 만들고, 이 데이터에 아까 설명한 로그-로그 피팅을 적용해서 "원래 정해둔 $\alpha$를 잘 복원해내는지" 확인해봅니다. 실제 모델 실험에 들어가기 전에, 분석 방법 자체가 잘 작동하는지 먼저 검증해보는 것입니다.

In [ ]:
# ---- 1단계: "정답을 알고 있는" 가짜 데이터 만들기 ----

np.random.seed(42)  # 항상 같은 결과가 나오도록 랜덤 시드 고정

# 모델 크기 N을 1천 개에서 1억 개까지, 자릿수가 하나씩 늘어나도록 6개 만듭니다.
toy_N = np.array([1e3, 1e4, 1e5, 1e6, 1e7, 1e8])

# 우리가 미리 정해둔 "진짜" 계수들. 나중에 이 값을 복원해낼 수 있는지 확인할 것입니다.
true_alpha = 0.15
true_C = 50.0

# 실제 실험에는 측정 노이즈가 섞이기 마련이므로, 아주 약간의 랜덤 노이즈를 곱해줍니다.
noise = np.random.normal(loc=0.0, scale=0.03, size=toy_N.shape)
toy_L = true_C * toy_N ** (-true_alpha) * np.exp(noise)

print("모델 크기 N:", toy_N)
print("Loss L:     ", np.round(toy_L, 4))

# ---- 2단계: 로그를 취하고, 직선으로 피팅(선형회귀)하기 ----

log_N = np.log10(toy_N)   # log10(모델 크기)
log_L = np.log10(toy_L)   # log10(Loss)

# np.polyfit(x, y, 1) : x, y 데이터에 "1차식(직선) y = m*x + b"를 가장 잘 맞도록 피팅합니다.
# 반환값은 [기울기(m), 절편(b)] 순서입니다.
coeffs = np.polyfit(log_N, log_L, 1)
slope, intercept = coeffs[0], coeffs[1]

# 위에서 유도했듯이 slope = -alpha 이므로, alpha = -slope 입니다.
estimated_alpha = -slope

print(f"\n우리가 미리 정한 진짜 alpha = {true_alpha}")
print(f"로그-로그 피팅으로 복원한 alpha = {estimated_alpha:.4f}")
print("-> 두 값이 거의 같다면, 피팅 방법 자체는 잘 작동한다는 뜻입니다.")

# ---- 3단계: 그래프로 확인 (선형 축 vs 로그-로그 축) ----

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 왼쪽: 있는 그대로(선형 축)의 그래프 -> 급격히 꺾이는 곡선으로 보입니다.
axes[0].plot(toy_N, toy_L, 'o-', color='tab:blue')
axes[0].set_xlabel('N (모델 크기, 파라미터 수)')
axes[0].set_ylabel('L (Loss)')
axes[0].set_title('① 원래 스케일 (선형 축)\n-> 곡선이라 규칙을 알아보기 어려움')

# 오른쪽: 로그-로그 축의 그래프 -> 직선으로 보여야 합니다.
axes[1].loglog(toy_N, toy_L, 'o', color='tab:red', markersize=8, label='가짜 데이터')
fit_line = 10 ** np.polyval(coeffs, log_N)  # 피팅된 직선을 다시 원래 스케일로 변환해서 그리기
axes[1].loglog(toy_N, fit_line, '--', color='gray', label=f'피팅 결과 (alpha={estimated_alpha:.3f})')
axes[1].set_xlabel('N (log 축)')
axes[1].set_ylabel('L (log 축)')
axes[1].set_title('② log-log 축\n-> 직선으로 보이면 Power Law가 맞다는 증거')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. 실험에서 사용할 용어(하이퍼파라미터) 미리 정리하기

이제부터 실제 Transformer 모델을 만들 텐데, 코드에 등장할 숫자(하이퍼파라미터)들이 각각 무엇을 의미하는지 먼저 정리하고 넘어가겠습니다. 이 표를 먼저 읽어두면 뒤에 나오는 코드가 훨씬 쉽게 읽힙니다.

| 이름 | 의미 | 비유 |
|---|---|---|
| `vocab_size` | 모델이 다룰 수 있는 서로 다른 토큰(단어 조각)의 개수 | 사전에 등록된 단어의 개수 |
| `seq_len` | 한 번에 모델에 입력하는 토큰의 개수 | 한 문장에 들어있는 단어 수 |
| `batch_size` | 한 번의 학습 스텝에서 동시에 처리하는 문장(시퀀스)의 개수 | 한 번에 채점하는 시험지 묶음 수 |
| `d_model` | 토큰 하나를 표현하는 벡터의 차원(길이) | 각 단어를 몇 개의 숫자로 "묘사"할지 (숫자가 많을수록 더 풍부하게 표현 가능) |
| `n_heads` | Self-Attention(자기 자신에게 주목하는 연산)을 몇 개의 독립적인 "시선"으로 나눠서 병렬로 계산할지 | 같은 문장을 동시에 여러 명이 서로 다른 관점으로 읽는 것 |
| `n_layers` | Transformer 인코더 블록을 몇 겹 쌓을지 | 같은 글을 몇 번 다시 읽으며 이해를 심화시키는지 |
| `d_ff` | 각 층 내부의 Feed-Forward(완전연결) 네트워크의 중간 차원. 보통 `d_model`의 4배로 설정 | 한 번 읽은 내용을 넓게 펼쳐서 다시 정리하는 메모장의 크기 |

이 중 `d_model`, `n_heads`, `n_layers`, `d_ff` 네 가지를 키우거나 줄이면 모델 전체의 파라미터 수(=모델 크기 $N$)가 달라집니다. 오늘 실험은 바로 이 네 가지를 조금씩 키워가면서 "모델 크기 $N$이 커질수록 Loss가 얼마나 좋아지는지"를 관찰하는 것입니다.

## 3. 미니 실습: 아주 작은 숫자로 shape의 흐름 따라가기

바로 큰 모델을 만들기 전에, **아주 작은 숫자**(장난감 크기)로 데이터가 모델 내부에서 어떤 모양(shape)으로 변해가는지 손으로 짚어보겠습니다. 이 노트북의 첫 번째 학습 목표("입력이 어떤 shape을 거쳐 출력이 되는지 추적하기")를 여기서 직접 확인합니다.

아래에서 사용하는 `toy_` 로 시작하는 값들은 오직 이해를 돕기 위한 장난감 숫자이며, 실제 실험에서 사용할 값과는 다릅니다.

In [ ]:
# 항상 같은 결과가 나오도록 시드를 고정합니다.
torch.manual_seed(0)

# 오직 이해를 돕기 위한 아주 작은 장난감 숫자들입니다.
toy_vocab_size = 20   # 서로 다른 토큰 20종류만 있다고 가정
toy_d_model = 8        # 각 토큰을 8개의 숫자로 표현
toy_seq_len = 6         # 한 문장에 토큰 6개
toy_batch_size = 2       # 문장 2개를 동시에 처리

# ---- 1) 입력: 토큰 ID ----
# 실제 문장이 아니라, 0 ~ (vocab_size-1) 사이의 정수로 이루어진 "토큰 ID"입니다.
# (진짜 문장이라면 "안녕/하세요/반갑/습니다" 같은 조각들이 미리 정수로 바뀌어 있다고 생각하면 됩니다.)
toy_token_ids = torch.randint(0, toy_vocab_size, (toy_batch_size, toy_seq_len))
print("1) 입력 토큰 ID (아직은 그냥 정수 나열일 뿐입니다)")
print("   shape:", tuple(toy_token_ids.shape), " # (batch_size, seq_len)")
print("   값:\n", toy_token_ids)

# ---- 2) 토큰 임베딩: 정수 ID -> 벡터 ----
# nn.Embedding(V, D)는 "V개의 정수 각각을 D차원 벡터에 대응시키는 조회 표(lookup table)"입니다.
toy_embedding_layer = nn.Embedding(toy_vocab_size, toy_d_model)
toy_token_vectors = toy_embedding_layer(toy_token_ids)
print("\n2) 토큰 임베딩 통과 후 (정수 ID -> d_model 차원의 벡터)")
print("   shape:", tuple(toy_token_vectors.shape), " # (batch_size, seq_len, d_model)")
print("   -> 각 토큰 하나하나가 이제 8개의 숫자로 이루어진 벡터가 되었습니다.")

# ---- 3) 위치(position) 번호 만들기 ----
# Transformer는 그 자체로는 "몇 번째 토큰인지" 순서 정보를 모릅니다.
# 그래서 0, 1, 2, ..., seq_len-1 이라는 위치 번호를 따로 만들어 위치 정보를 알려줘야 합니다.
toy_positions = torch.arange(toy_seq_len)
print("\n3) 위치 번호 생성: 0번째, 1번째, ... 토큰이라는 표시")
print("   shape:", tuple(toy_positions.shape), " 값:", toy_positions)

# ---- 4) 위치 임베딩: 위치 번호 -> 벡터 ----
# 토큰 임베딩과 똑같은 방식으로, "몇 번째 위치인지"도 벡터로 바꿔줍니다.
toy_pos_emb_layer = nn.Embedding(512, toy_d_model)  # 최대 512개 위치까지 지원한다고 가정
toy_pos_vectors = toy_pos_emb_layer(toy_positions)
print("\n4) 위치 임베딩 통과 후")
print("   shape:", tuple(toy_pos_vectors.shape), " # (seq_len, d_model)")

# ---- 5) 두 벡터를 더하기 (브로드캐스팅) ----
# (batch, seq, d_model) 모양의 토큰 벡터에 (seq, d_model) 모양의 위치 벡터를 더하면,
# 파이토치가 자동으로 배치 차원에 맞춰 똑같은 위치 벡터를 복사해서 더해줍니다. (이를 "브로드캐스팅"이라고 합니다.)
toy_combined = toy_token_vectors + toy_pos_vectors
print("\n5) 토큰 벡터 + 위치 벡터")
print("   shape:", tuple(toy_combined.shape), " # 여전히 (batch_size, seq_len, d_model)")
print("   -> 이제 각 벡터에는 '무슨 토큰인지'와 '몇 번째 토큰인지' 정보가 모두 들어있습니다.")

# ---- 6) (여기서는 생략) Transformer 인코더 ----
# 실제 모델에서는 이 다음에 Self-Attention + Feed-Forward로 이루어진 인코더를 통과합니다.
# 미리 스포일러를 드리면: 인코더를 통과해도 shape은 그대로 (batch, seq_len, d_model)로 유지됩니다.
# (내부적으로 "각 토큰이 서로를 얼마나 참고할지" 계산이 일어나지만, 텐서의 모양 자체는 바뀌지 않습니다.)
# 인코더의 자세한 동작은 바로 다음 섹션(Causal Mask)과 이후 모델 클래스 정의에서 다시 다룹니다.

# ---- 7) 마지막 Linear층(head): d_model -> vocab_size ----
# 지금까지는 각 위치가 "의미를 담은 벡터"였다면, 이제 이 벡터를 다시
# "vocab_size개의 토큰 중 다음에 무엇이 올 확률이 높은지"를 나타내는 숫자들로 바꿔야 합니다.
toy_head = nn.Linear(toy_d_model, toy_vocab_size)
toy_logits = toy_head(toy_combined)  # 인코더를 통과했다고 가정하고, combined에 바로 head를 적용
print("\n6) 마지막 Linear층(head) 통과 후")
print("   shape:", tuple(toy_logits.shape), " # (batch_size, seq_len, vocab_size)")
print("   -> 각 위치마다 vocab_size(=20)개의 점수(logit)가 생겼습니다.")
print("      이 점수에 softmax를 취하면 '다음 토큰 확률분포'가 됩니다.")

## 4. 다음 토큰을 예측할 때 "미래를 훔쳐보면" 안 되는 이유 — Causal Mask

우리가 만들 모델은 **"지금까지 나온 토큰들을 보고 다음 토큰을 맞히기"** 를 학습합니다. 국어 시험에 비유하면, "이 문장 다음에 올 단어를 맞혀보세요"라는 문제를 푸는 것과 같습니다.

그런데 만약 시험 문제 옆에 정답이 이미 인쇄되어 있다면 어떨까요? 그 문제는 아무 의미가 없어집니다 — 실력을 전혀 기르지 못한 채 "정답 위치를 베끼는 요령"만 배우게 됩니다.

Transformer의 Self-Attention은 기본적으로 입력 시퀀스의 **모든 위치를 서로 자유롭게 참고**할 수 있습니다(이를 양방향(bidirectional)이라고 부릅니다). 그런데 우리 학습 방식에서는 $i$번째 위치의 출력으로 $(i+1)$번째 토큰(=바로 다음 정답)을 예측합니다. 만약 아무 제한이 없다면, $i$번째 위치는 입력에 이미 들어있는 $(i+1)$번째 토큰 벡터를 그냥 "참고"해서 그대로 베껴버릴 수 있습니다 — 정답이 시험지에 미리 적혀 있는 것과 같은 상황이 벌어지는 것입니다.

이 문제를 막기 위해 **Causal Mask(인과적 마스크)** 를 사용합니다. Causal Mask는 "각 위치는 자기 자신과 그 이전 위치만 볼 수 있고, 이후(미래) 위치는 절대 볼 수 없다"는 규칙을 강제로 적용하는 장치입니다. 이렇게 하면 모델은 진짜로 "이전 문맥으로부터 다음 토큰을 예측하는 법"을 배울 수밖에 없습니다.

아래에서 이 마스크가 실제로 어떤 모양인지 직접 만들어서 확인해보겠습니다.

In [ ]:
# PyTorch에는 Causal Mask를 만들어주는 함수가 이미 준비되어 있습니다.
# 이해를 돕기 위해 아주 작은 5x5 크기(문장 길이 5라고 가정)로 만들어보겠습니다.
demo_seq_len = 5
demo_mask = nn.Transformer.generate_square_subsequent_mask(demo_seq_len)
print("생성된 마스크 (5x5):")
print(demo_mask)

print("""
읽는 법:
  - 행(row)    = "지금 예측하려는 위치" (몇 번째 토큰을 예측 중인지)
  - 열(column) = "참고하려는 위치"     (몇 번째 토큰을 들여다보려 하는지)
  - 값이 0     -> "봐도 됩니다" (참고 허용)
  - 값이 -inf  -> "보면 안 됩니다" (참고 금지)

예를 들어 0번째 행(0번째 토큰을 예측할 때)은 [0, -inf, -inf, -inf, -inf] 이므로
0번째 위치(자기 자신)만 볼 수 있고, 1~4번째(미래)는 전부 차단됩니다.

2번째 행(2번째 토큰을 예측할 때)은 [0, 0, 0, -inf, -inf] 이므로
0, 1, 2번째(자기 자신까지의 과거)는 볼 수 있고, 3, 4번째(미래)는 차단됩니다.
""")

# 왜 하필 -inf일까요? Self-Attention 내부에서는 이 마스크를 attention score(관심도 점수)에 "더한" 뒤
# softmax를 취해서 확률처럼 바꿉니다. -inf를 더하면 그 위치의 score가 음의 무한대가 되고,
# softmax(음의 무한대) = 0 이 되어 "그 위치를 참고할 확률이 정확히 0%" 가 됩니다.
example_scores = torch.tensor([2.0, 1.0, 0.0, -1.0, -2.0])
masked_scores = example_scores + demo_mask[2]  # 2번째 위치(토큰)를 예측한다고 가정
print("원래 attention score:      ", example_scores)
print("2번째 위치용 마스크를 더한 후:", masked_scores)
print("softmax 적용 후 (확률로 변환):", torch.softmax(masked_scores, dim=0).round(decimals=3))
print("-> 3, 4번째(미래) 위치의 확률이 정확히 0이 된 것을 확인할 수 있습니다.")

## 5. 이제 모델 클래스를 만들어봅시다

지금까지 배운 내용을 하나로 합쳐서 실제 `TinyTransformer` 클래스를 정의합니다. 앞서 미니 실습(3장)에서 손으로 짚어본 "임베딩 → 위치 임베딩 → (인코더) → head" 흐름에, 4장에서 배운 Causal Mask까지 추가합니다.

원본 코드와 달라진 점을 미리 요약하면:

1. **(들여쓰기 오류 수정)** 원본 코드는 클래스 내부 들여쓰기가 어긋나 있어 그대로 실행하면 `IndentationError`가 발생합니다. 아래 코드는 이를 바로잡았습니다.
2. **(핵심 수정) Causal Mask 추가**: 원본 코드는 `self.encoder(x)`만 호출해서 마스크가 전혀 적용되지 않았습니다 — 바로 앞 장에서 설명한 "미래를 훔쳐보는" 문제가 그대로 있었던 것입니다. 아래 코드는 `use_causal_mask`라는 옵션을 추가해서, 이 옵션을 켜고 끄며 직접 그 차이를 실험해볼 수 있게 했습니다. (기본값은 `True`, 즉 마스크를 사용하는 올바른 설정입니다.)
3. **(개선) device 지원**: GPU가 있으면 자동으로 GPU에서 계산하도록 `device`를 명시적으로 다룹니다.

In [ ]:
class TinyTransformer(nn.Module):
    """
    아주 작게 만들 수 있는 GPT 스타일(다음 토큰 예측) Transformer 모델입니다.
    d_model, n_heads, n_layers, d_ff 값을 바꾸면 모델 크기(파라미터 수)가 달라집니다.
    """

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff=None, use_causal_mask=True):
        super().__init__()

        # d_ff를 따로 지정하지 않으면 관례적으로 d_model의 4배를 사용합니다.
        d_ff = d_ff or d_model * 4

        # 뒤에서(4장의 실험) 이 값을 True/False로 바꿔가며 실험할 수 있도록 저장해둡니다.
        self.use_causal_mask = use_causal_mask

        # 3장에서 손으로 짚어본 것과 동일한 두 개의 임베딩 층입니다.
        self.embedding = nn.Embedding(vocab_size, d_model)   # 토큰 ID -> 벡터
        self.pos_emb = nn.Embedding(512, d_model)             # 위치 번호 -> 벡터 (최대 512 토큰까지 지원)

        # Self-Attention + Feed-Forward로 이루어진 인코더 블록 하나를 정의하고,
        # 이를 n_layers만큼 쌓습니다. (n_layers가 클수록 더 "깊은" 모델이 됩니다.)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=0.1,            # 과적합을 줄이기 위해 학습 중 일부 뉴런을 무작위로 끕니다.
            batch_first=True,       # 입력 shape을 (batch, seq_len, d_model) 순서로 사용하겠다는 의미
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # 3장에서 본 것과 동일한 마지막 Linear층: d_model 차원 벡터 -> vocab_size개의 점수(logit)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len) — 토큰 ID로 이루어진 정수 텐서

        seq_len = x.size(1)
        pos = torch.arange(seq_len, device=x.device)  # 0, 1, ..., seq_len-1

        # 3장에서 손으로 했던 것과 동일: 토큰 벡터 + 위치 벡터
        x = self.embedding(x) + self.pos_emb(pos)  # -> (batch_size, seq_len, d_model)

        if self.use_causal_mask:
            # 4장에서 만들어본 것과 같은 마스크를 이 시퀀스 길이에 맞게 생성합니다.
            # is_causal=True는 "이 mask가 causal(미래를 가리는) 마스크입니다"라는 힌트로,
            # PyTorch가 가능한 경우 더 빠른 연산 경로를 사용할 수 있게 해줍니다.
            causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
            x = self.encoder(x, mask=causal_mask, is_causal=True)
        else:
            # (일부러 버그를 재현해보고 싶을 때) 마스크 없이 인코더만 통과시킵니다.
            # -> 4장에서 설명한 "미래를 훔쳐보는" 문제가 그대로 발생합니다.
            x = self.encoder(x)

        # shape은 여전히 (batch_size, seq_len, d_model) — 인코더는 shape을 바꾸지 않습니다.
        return self.head(x)  # -> (batch_size, seq_len, vocab_size)

### 5-1. 작은 모델로 직접 shape 확인해보기

클래스를 정의했으니, 3장에서 사용했던 것과 **똑같은 장난감 크기**로 실제 `TinyTransformer`를 만들어서 forward를 한 번 실행해보고, shape이 우리가 예상한 대로 나오는지 확인해보겠습니다.

In [ ]:
torch.manual_seed(0)

# 3장에서 쓴 것과 동일한 장난감 크기
toy_model = TinyTransformer(
    vocab_size=toy_vocab_size,
    d_model=toy_d_model,
    n_heads=2,
    n_layers=2,
    use_causal_mask=True
)

toy_output = toy_model(toy_token_ids)  # 3장에서 만든 toy_token_ids를 그대로 재사용

print("입력 shape :", tuple(toy_token_ids.shape), " # (batch_size, seq_len)")
print("출력 shape :", tuple(toy_output.shape), " # (batch_size, seq_len, vocab_size)")

expected_shape = (toy_batch_size, toy_seq_len, toy_vocab_size)
assert tuple(toy_output.shape) == expected_shape
print(f"\n예상했던 shape {expected_shape} 과 정확히 일치합니다!")
print("-> 3장에서 손으로 따라가 본 흐름(임베딩 -> 위치 임베딩 -> 인코더 -> head)이")
print("   실제 클래스 안에서도 동일하게 동작하고 있다는 것을 확인했습니다.")

n_toy_params = sum(p.numel() for p in toy_model.parameters())
print(f"\n참고: 이 장난감 모델의 파라미터 수는 {n_toy_params:,}개입니다.")
print("(실제 실험에서는 이보다 훨씬 큰 모델들을 사용합니다.)")

## 6. 실험에서 흔히 빠지는 함정 두 가지

본격적으로 "모델 크기별 실험"에 들어가기 전에, Scaling Law 실험을 설계할 때 흔히 발생하는 두 가지 문제를 **일부러 직접 재현**해보면서 왜 문제가 되는지 체감해보겠습니다.

- **함정 1**: Causal Mask 없이 학습하면? (4장에서 배운 내용의 실전 확인)
- **함정 2**: 학습 데이터에 아무런 규칙(패턴)이 없다면?

두 실험 모두 **같은 학습용 데이터 뭉치**를 사용할 것이므로, 먼저 이 데이터부터 준비하겠습니다.

In [ ]:
# ---- 실험용 데이터 준비: "반복해서 등장하는 문장 모음" 만들기 ----
#
# 진짜 텍스트 데이터에는 문법, 자주 쓰이는 표현, 반복되는 패턴이 존재합니다.
# 이런 "배울 거리(규칙)"가 있어야 모델이 클수록 더 잘 학습할 수 있습니다.
#
# 여기서는 진짜 텍스트 대신, 아주 단순화된 가짜 데이터를 사용합니다:
# 무작위 토큰으로 이루어진 "시퀀스(문장)" 300개를 미리 만들어두고,
# 학습 중에는 이 300개 중에서 반복적으로 뽑아서 사용합니다.
# 마치 300문장짜리 짧은 교과서를 여러 번 반복해서 읽으며 공부하는 것과 비슷합니다.
# (완전히 새로운 문장이 매번 등장하는 게 아니라, "정해진 범위 안의 패턴"을 반복 학습하는 것이 핵심입니다.)

vocab_size = 100   # 이번 장부터 끝까지 사용할 공통 설정
seq_len = 32
batch_size = 16
N_TEMPLATES = 300   # 미리 만들어둘 "문장" 개수

torch.manual_seed(777)  # 이 시드를 고정해서, 모든 실험이 "동일한 교과서"를 사용하도록 합니다.
template_pool = torch.randint(0, vocab_size, (N_TEMPLATES, seq_len + 1))
# seq_len + 1인 이유: 앞의 seq_len개는 "입력"으로, 뒤의 seq_len개(한 칸 밀린 것)는 "정답"으로 사용하기 때문입니다.
# (7장의 학습 루프에서 x[:, :-1]을 입력으로, x[:, 1:]을 정답으로 사용하는 것과 연결됩니다.)

print(f"template_pool shape: {tuple(template_pool.shape)}  # (N_TEMPLATES, seq_len + 1)")
print(f"이론적 최소 Loss (완전히 무작위로 찍었을 때) = ln(vocab_size) = {math.log(vocab_size):.4f}")
print("-> 이 숫자는 뒤에서 계속 '기준선(하한선)'으로 등장할 것이므로 기억해두세요.")

### 6-1. 함정 1: Causal Mask가 없으면 어떻게 될까?

4장에서 "마스크가 없으면 미래를 훔쳐볼 수 있다"고 설명했습니다. 이번에는 정말 그런 일이 일어나는지 직접 학습을 시켜서 확인해보겠습니다.

**실험 설계**: 모델 크기와 데이터는 완전히 동일하게 고정하고, `use_causal_mask` 옵션만 `True`/`False`로 바꿔서 비교합니다. 이렇게 **딱 한 가지 조건만 바꾸고 나머지는 전부 고정**하는 것이 공정한 비교를 위한 기본 원칙입니다.

In [ ]:
def train_small_experiment(use_causal_mask, n_steps=200, base_lr=1e-3, warmup_steps=20, seed=0):
    """
    같은 크기의 모델을 template_pool 데이터로 n_steps만큼 학습시키고,
    각 스텝의 loss 기록을 리스트로 반환하는 헬퍼 함수입니다.
    (이 함수는 6장의 두 실험, 7장의 본 실험에서 공통으로 재사용합니다.)
    """
    torch.manual_seed(seed)
    model = TinyTransformer(
        vocab_size, d_model=96, n_heads=4, n_layers=4, d_ff=384,
        use_causal_mask=use_causal_mask
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.01)
    # Warmup: 학습 초반 warmup_steps 동안은 학습률을 0에서 base_lr까지 서서히 끌어올립니다.
    # 이렇게 하면 학습 초반의 불안정한 큰 업데이트를 방지할 수 있습니다.
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: min(1.0, (step + 1) / warmup_steps)
    )

    model.train()
    losses = []
    for step in range(n_steps):
        idx = torch.randint(0, N_TEMPLATES, (batch_size,))
        x = template_pool[idx].to(device)

        logits = model(x[:, :-1])  # 마지막 토큰을 제외한 나머지를 입력으로
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, vocab_size),
            x[:, 1:].reshape(-1)  # 첫 토큰을 제외한 나머지(한 칸 밀린 것)를 정답으로
        )

        optimizer.zero_grad()
        loss.backward()
        # 그래디언트가 너무 커지는 것을 막아 학습을 안정화합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    return losses


print("=== mask 없음 (4장에서 설명한 '미래 훔쳐보기' 버그 재현) ===")
losses_no_mask = train_small_experiment(use_causal_mask=False)
print(f"  1 스텝: {losses_no_mask[0]:.3f}   "
      f"50 스텝: {losses_no_mask[49]:.3f}   "
      f"100 스텝: {losses_no_mask[99]:.3f}   "
      f"200 스텝: {losses_no_mask[199]:.3f}")

print("\n=== mask 적용 (4장에서 배운 대로 올바르게 수정) ===")
losses_with_mask = train_small_experiment(use_causal_mask=True)
print(f"  1 스텝: {losses_with_mask[0]:.3f}   "
      f"50 스텝: {losses_with_mask[49]:.3f}   "
      f"100 스텝: {losses_with_mask[99]:.3f}   "
      f"200 스텝: {losses_with_mask[199]:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses_no_mask, label='mask 없음 (버그)', color='tab:red')
ax.plot(losses_with_mask, label='mask 적용 (올바름)', color='tab:blue')
ax.axhline(math.log(vocab_size), color='gray', linestyle=':', label='이론적 하한 ln(vocab_size)')
ax.set_xlabel('학습 스텝')
ax.set_ylabel('Loss')
ax.set_title('Causal Mask 유무에 따른 학습 곡선 비교')
ax.legend()
plt.tight_layout()
plt.show()

**결과 해석**

학습 초반(예: 50 스텝 근처)에는 두 곡선이 비슷하게 시작하지만, 학습이 진행될수록 **mask가 없는 쪽의 Loss가 비정상적으로 빠르게 0에 가까워지는** 것을 볼 수 있습니다.

이는 모델의 능력이 좋아져서가 아니라, 모델이 "입력 안에 이미 들어있는 정답(바로 다음 위치의 토큰)을 그대로 베끼는" 지름길을 찾아냈기 때문입니다. 4장에서 설명한 "정답이 시험지에 미리 적혀 있는" 상황이 실제로 벌어진 것입니다.

반면 mask가 적용된 쪽은 Loss가 더 천천히, 그러나 "정직하게" 줄어듭니다 — 진짜로 이전 문맥에서 다음 토큰의 패턴을 학습하고 있기 때문입니다.

> 이 문제는 모델 크기와 무관하게 발생합니다. 즉, mask가 없는 상태로 7장의 "모델 크기별 비교" 실험을 했다면, 크기와 상관없이 모든 모델이 비슷하게 낮은(비정상적인) Loss를 내서 Scaling Law를 제대로 관찰할 수 없었을 것입니다.

### 6-2. 함정 2: 데이터에 아무 패턴이 없다면?

원본 코드는 매 학습 스텝마다 **완전히 새로운 무작위 데이터**(`torch.randint`)를 생성해서 사용했습니다. 얼핏 보면 "매번 새로운 데이터로 학습하니까 더 좋지 않을까?"라고 생각할 수도 있지만, 사실 이렇게 하면 데이터에 애초에 배울 수 있는 규칙이 전혀 없다는 문제가 있습니다.

**왜 문제가 될까요?** Cross-entropy Loss는 "모델이 다음 토큰의 확률분포를 얼마나 정확히 맞히는가"를 나타냅니다. 만약 정답 토큰이 완전히 무작위이고 `vocab_size`개의 토큰 중 하나가 균등한 확률로 나온다면, 세상 어떤 모델도(아무리 크더라도!) 평균적으로 다음 값보다 낮은 Loss를 낼 수 없습니다.

$$L_{\text{min}} = \ln(\text{vocab\_size})$$

정말로 예측할 수 있는 정보 자체가 데이터 안에 없기 때문입니다. 이 개념은 Hoffmann et al. (2022, 이른바 'Chinchilla' 논문)에서 다음과 같이 명시적으로 표현되었습니다.

$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}$$

여기서 $E$는 데이터 자체가 가진 근본적인 불확실성(엔트로피)으로, **모델을 아무리 키워도(N을 키워도) 줄어들지 않는 부분**입니다. 이런 이유로 이를 **irreducible loss(줄일 수 없는 손실)** 라고 부릅니다. 순수 무작위 데이터를 사용하면 사실상 $E = \ln(\text{vocab\_size})$이고 $A/N^{\alpha}$ 항으로 개선될 여지가 없는 것과 마찬가지이므로, 모델을 키워도 Loss가 줄어들 수 없는 것입니다.

**실험 설계**: 이번에는 모델 크기와 mask 사용 여부는 동일하게 고정하고, **데이터를 만드는 방식만** 바꿔서 비교합니다.
- "매번 새로운 무작위 데이터": 원본 코드와 동일한 방식
- "고정된 템플릿 풀 재사용": 앞서 6장 초반에 준비한 `template_pool`을 반복 사용

In [ ]:
def train_data_comparison(data_mode, n_steps=200, base_lr=1e-3, warmup_steps=20, seed=0):
    """data_mode: 'fresh_random' 이면 매번 새 무작위 데이터, 'template' 이면 template_pool 재사용."""
    torch.manual_seed(seed)
    model = TinyTransformer(
        vocab_size, d_model=96, n_heads=4, n_layers=4, d_ff=384,
        use_causal_mask=True  # 이번에는 mask를 항상 켜서, '데이터' 조건만 바뀌도록 통제합니다.
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: min(1.0, (step + 1) / warmup_steps)
    )

    model.train()
    losses = []
    for step in range(n_steps):
        if data_mode == 'fresh_random':
            # 원본 코드와 동일한 방식: 매 스텝 완전히 새로운 무작위 토큰을 생성합니다.
            x = torch.randint(0, vocab_size, (batch_size, seq_len + 1)).to(device)
        else:
            # template_pool 300개 중 batch_size개를 골라서 재사용합니다.
            idx = torch.randint(0, N_TEMPLATES, (batch_size,))
            x = template_pool[idx].to(device)

        logits = model(x[:, :-1])
        loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), x[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    return losses


print(f"이론적 최소값 ln(vocab_size) = {math.log(vocab_size):.4f}\n")

print("=== 매번 새로운 무작위 데이터 (원본 코드와 동일한 방식) ===")
losses_fresh = train_data_comparison('fresh_random')
print(f"  1 스텝: {losses_fresh[0]:.3f}   50 스텝: {losses_fresh[49]:.3f}   "
      f"100 스텝: {losses_fresh[99]:.3f}   200 스텝: {losses_fresh[199]:.3f}")

print("\n=== 고정된 템플릿 풀 재사용 (수정된 방식) ===")
losses_template = train_data_comparison('template')
print(f"  1 스텝: {losses_template[0]:.3f}   50 스텝: {losses_template[49]:.3f}   "
      f"100 스텝: {losses_template[99]:.3f}   200 스텝: {losses_template[199]:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses_fresh, label='매번 새로운 무작위 데이터', color='tab:orange')
ax.plot(losses_template, label='고정된 템플릿 풀 재사용', color='tab:green')
ax.axhline(math.log(vocab_size), color='gray', linestyle=':', label='이론적 하한 ln(vocab_size)')
ax.set_xlabel('학습 스텝')
ax.set_ylabel('Loss')
ax.set_title('데이터에 반복되는 패턴이 있는지 여부에 따른 학습 곡선 비교\n(모델 크기·mask 설정은 동일)')
ax.legend()
plt.tight_layout()
plt.show()

**결과 해석**

"매번 새로운 무작위 데이터"로 학습한 곡선은 이론적 하한선(`ln(vocab_size)`) 근처에서 거의 평평하게 유지됩니다 — 학습이 진행되어도 별로 좋아지지 않습니다. 데이터 자체에 배울 수 있는 규칙이 없으니, 모델이 아무리 애써도 무작위로 찍는 수준 이상으로는 잘할 수가 없는 것입니다.

반면 "고정된 템플릿 풀 재사용" 곡선은 꾸준히 아래로 내려갑니다 — 같은 300개의 패턴이 반복해서 등장하니, 모델이 그 패턴을 점점 더 잘 기억하고 예측할 수 있게 되는 것입니다.

> 이 실험이 시사하는 바: 만약 원본 코드처럼 순수 무작위 데이터로 "모델 크기별 비교" 실험을 했다면, 모델을 아무리 키워도 Loss가 `ln(vocab_size)` 근처에서 벗어나지 못했을 것이고, 그러면 애초에 비교할 만한 "차이"가 거의 나타나지 않아 Power Law를 확인하는 것 자체가 불가능했을 것입니다.

이제 두 함정을 모두 확인했으니, 7장에서는 **mask를 올바르게 적용**하고 **패턴이 있는 데이터**를 사용해서, 진짜로 모델 크기별 실험을 진행해보겠습니다.

## 7. 진짜 실험: 5가지 크기의 모델로 Scaling Law 검증하기

이제 준비가 끝났습니다. Causal Mask를 올바르게 적용하고, 반복되는 패턴이 있는 `template_pool` 데이터를 사용해서, **크기가 다른 5개의 모델**을 학습시켜 보겠습니다.

| 이름 | d_model | n_heads | n_layers | d_ff |
|---|---|---|---|---|
| XS | 32  | 2 | 2 | 128  |
| S  | 64  | 4 | 3 | 256  |
| M  | 96  | 4 | 4 | 384  |
| L  | 160 | 8 | 6 | 640  |
| XL | 256 | 8 | 8 | 1024 |

**공정한 비교를 위한 원칙**: 6장에서 두 가지 함정을 실험할 때처럼, 이번에도 "모델 크기"만 바뀌고 나머지 조건(mask 사용 여부, 데이터, 학습 스텝 수, 학습률, warmup 등)은 모두 동일하게 고정합니다. 심지어 **각 모델이 학습 중 보게 되는 데이터의 순서**까지도 동일하게 맞춰서, 우연히 "더 쉬운 배치를 뽑은 모델"이 유리해지는 일이 없도록 통제합니다. (아래 코드에서 데이터용 시드를 모델 크기와 무관하게 고정하는 부분이 이 역할을 합니다.)

**예상 소요 시간**: CPU에서 전체 실행에 약 2~3분 정도 걸립니다 (모델이 클수록 한 스텝에 걸리는 시간도 늘어나므로, XL 모델이 가장 오래 걸립니다). GPU가 있는 환경(Colab 등)에서는 훨씬 빠르게 끝납니다.

In [ ]:
# 위 표와 동일한 설정입니다. d_ff는 관례대로 d_model의 4배로 맞췄습니다.
scaling_configs = [
    {"name": "XS", "d_model": 32,  "n_heads": 2, "n_layers": 2, "d_ff": 128},
    {"name": "S",  "d_model": 64,  "n_heads": 4, "n_layers": 3, "d_ff": 256},
    {"name": "M",  "d_model": 96,  "n_heads": 4, "n_layers": 4, "d_ff": 384},
    {"name": "L",  "d_model": 160, "n_heads": 8, "n_layers": 6, "d_ff": 640},
    {"name": "XL", "d_model": 256, "n_heads": 8, "n_layers": 8, "d_ff": 1024},
]

# 각 설정으로 만들어질 모델의 대략적인 파라미터 수를 미리 살펴봅니다.
print("설정별 예상 파라미터 수:")
for config in scaling_configs:
    preview_model = TinyTransformer(vocab_size, **{k: v for k, v in config.items() if k != 'name'})
    n_params = sum(p.numel() for p in preview_model.parameters())
    print(f"  {config['name']:3s}: {n_params:>10,} 개")
    del preview_model  # 파라미터 수만 확인하고 바로 메모리에서 정리합니다 (실제 학습은 다음 셀에서).

In [ ]:
n_steps = 200
base_lr = 1e-3
warmup_steps = 20  # 전체 스텝의 10% 정도를 warmup으로 사용하는 것이 일반적인 관례입니다.

results = []       # (이름, 파라미터 수, 마지막 10스텝 평균 loss, 걸린 시간) 튜플을 저장
all_losses = {}     # 이름 -> 전체 학습 곡선(loss 리스트). 나중에 학습 곡선을 그릴 때 사용합니다.

print("=== Scaling Law 실험: 모델 크기별 Loss 변화 ===\n")
print(f"이론적 최소값 ln(vocab_size) = {math.log(vocab_size):.4f}\n")

for config in scaling_configs:
    # 가중치 초기화용 시드는 매번 동일(0)하게 두어, 결과가 재현 가능하도록 합니다.
    torch.manual_seed(0)
    model = TinyTransformer(
        vocab_size, **{k: v for k, v in config.items() if k != 'name'}, use_causal_mask=True
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: min(1.0, (step + 1) / warmup_steps)
    )

    # 데이터를 뽑는 순서용 시드는 "모델 크기와 무관하게" 항상 동일한 값(2024)으로 고정합니다.
    # 이렇게 하면 XS든 XL이든 학습 중 정확히 같은 순서로 같은 배치를 보게 되어,
    # "운 좋게 쉬운 데이터를 뽑았다"는 우연이 결과에 섞여 들어가지 않습니다.
    data_generator = torch.Generator().manual_seed(2024)

    model.train()
    losses = []
    start_time = time.time()

    for step in range(n_steps):
        idx = torch.randint(0, N_TEMPLATES, (batch_size,), generator=data_generator)
        x = template_pool[idx].to(device)

        logits = model(x[:, :-1])
        loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), x[:, 1:].reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    elapsed = time.time() - start_time
    final_loss = float(np.mean(losses[-10:]))  # 마지막 한 스텝은 노이즈가 있을 수 있어, 최근 10스텝 평균을 사용합니다.

    results.append((config['name'], n_params, final_loss, elapsed))
    all_losses[config['name']] = losses

    print(f"  {config['name']:3s}: {n_params:>10,} params -> "
          f"Loss: {final_loss:.4f} (마지막 스텝: {losses[-1]:.4f}), Time: {elapsed:.1f}s")

print("\n모든 모델의 학습이 끝났습니다.")

## 8. 결과 정리 및 Power Law 피팅

이제 5개 모델의 결과를 표로 정리하고, 1장에서 연습했던 것과 **똑같은 방법**(로그-로그 축에서 직선 피팅)으로 실제 데이터에서 $\alpha$를 추정해보겠습니다.

In [ ]:
print(f"{'모델':^6} | {'파라미터 수':>12} | {'최종 Loss':>10} | {'학습 시간':>10}")
print("-" * 50)
for name, n_params, final_loss, elapsed in results:
    print(f"{name:^6} | {n_params:>12,} | {final_loss:>10.4f} | {elapsed:>9.1f}s")

In [ ]:
# 1장에서 연습했던 것과 완전히 동일한 방법입니다:
#   1) N과 L에 각각 log10을 취하고
#   2) np.polyfit으로 직선(1차식)을 피팅하고
#   3) 기울기에 마이너스를 붙이면 alpha

Ns = np.array([r[1] for r in results], dtype=float)   # 파라미터 수
Ls = np.array([r[2] for r in results], dtype=float)   # 최종 Loss
names = [r[0] for r in results]

log_N = np.log10(Ns)
log_L = np.log10(Ls)

coeffs = np.polyfit(log_N, log_L, 1)
alpha_estimated = -coeffs[0]

# 피팅이 데이터를 얼마나 잘 설명하는지(R^2, 1에 가까울수록 직선에 가깝다는 뜻)도 계산해봅니다.
predicted = np.polyval(coeffs, log_N)
ss_res = np.sum((log_L - predicted) ** 2)
ss_tot = np.sum((log_L - log_L.mean()) ** 2)
r_squared = 1 - ss_res / ss_tot

print(f"우리 실험에서 추정한 alpha_N = {alpha_estimated:.3f}")
print(f"log-log 축에서의 R^2 (직선에 얼마나 가까운지) = {r_squared:.3f}")
print(f"\n(참고) Kaplan et al. (2020) 논문에서 보고한 alpha_N = 0.076")
print("-> 우리 실험값과 논문 값이 다른 것은 지극히 정상입니다. 이유는 9장에서 자세히 설명합니다.")

### 8-1. 그래프로 직접 확인하기

숫자로 된 표보다 그래프로 보면 훨씬 직관적으로 이해할 수 있습니다. 두 가지 그래프를 그려보겠습니다.

1. **학습 곡선**: 5개 모델이 각각 200 스텝 동안 어떻게 학습되어 갔는지 (작은 모델은 일찍 한계에 부딪히고, 큰 모델은 계속 좋아지는 모습을 볼 수 있는지 확인)
2. **최종 Power Law 그래프**: 1장의 가짜 데이터 실험과 동일한 형식으로, 이번에는 진짜 실험 결과를 사용

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for name, losses in all_losses.items():
    ax.plot(losses, label=name)
ax.axhline(math.log(vocab_size), color='gray', linestyle=':', linewidth=1.5, label='이론적 하한 ln(vocab_size)')
ax.set_xlabel('학습 스텝')
ax.set_ylabel('Loss')
ax.set_title('모델 크기별 학습 곡선\n(작은 모델은 일찍 한계에 부딪히고, 큰 모델일수록 계속 좋아지는지 확인해보세요)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: 선형 축
axes[0].plot(Ns, Ls, 'o-', color='tab:blue', markersize=8)
for n, l, name in zip(Ns, Ls, names):
    axes[0].annotate(name, (n, l), textcoords="offset points", xytext=(6, 6))
axes[0].set_xlabel('N (파라미터 수)')
axes[0].set_ylabel('L (최종 Loss)')
axes[0].set_title('① 원래 스케일 (선형 축)')

# 오른쪽: 로그-로그 축 + 피팅한 직선
axes[1].loglog(Ns, Ls, 'o', color='tab:blue', markersize=9, label='실험 결과 (5개 모델)')
fit_curve = 10 ** np.polyval(coeffs, np.log10(Ns))
axes[1].loglog(Ns, fit_curve, '--', color='tab:red', label=f'피팅된 Power Law (alpha={alpha_estimated:.3f})')
for n, l, name in zip(Ns, Ls, names):
    axes[1].annotate(name, (n, l), textcoords="offset points", xytext=(6, 6))
axes[1].set_xlabel('N (log 축)')
axes[1].set_ylabel('L (log 축)')
axes[1].set_title('② log-log 축\n-> 점들이 대략 직선을 이루면 Power Law 확인!')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. 결과 해석하기: 왜 우리 alpha는 논문 값(0.076)과 다를까?

방금 얻은 $\alpha$ 값이 Kaplan et al. (2020) 논문의 0.076과 상당히 다르게 나왔을 것입니다. 이는 계산이 잘못된 것이 아니라, 우리 실험과 논문의 실험이 여러 면에서 **근본적으로 다른 규모와 설정**이기 때문입니다. 아래 차이점들을 알아두면, 앞으로 비슷한 "작은 재현 실험"의 결과를 해석할 때 도움이 됩니다.

| 항목 | 논문 (Kaplan et al., 2020) | 이 노트북의 실험 |
|---|---|---|
| 데이터 | 실제 방대한 텍스트 (WebText2 등) | 300개 시퀀스를 반복하는 단순화된 가짜 데이터 |
| 모델 크기 범위 | 여러 자릿수(orders of magnitude)에 걸친 매우 넓은 범위 | 약 5만 ~ 650만 파라미터, 상대적으로 좁은 범위 |
| 학습 스텝 | 수렴할 때까지 충분히 학습 | 200 스텝으로 제한 (특히 큰 모델은 아직 다 수렴하지 못했을 수 있음) |
| 반복 실험 | 여러 설정·시드로 안정적인 추세 확인 | 한 번의 시드로만 실행 (시드를 바꾸면 alpha 값도 어느 정도 달라집니다) |
| 파라미터 집계 방식 | 임베딩을 제외한 파라미터 수 기준 | 전체 파라미터 수 기준 |

또한, 우리 데이터는 실제 언어가 아니라 "정해진 300개 문장을 외우는" 것에 가까운 **암기(memorization) 성격이 강한 과제**입니다. 실제 언어 모델링(다양한 새로운 문장에 대한 일반화)과는 다른 종류의 학습이므로, 이 역시 지수가 다르게 나오는 이유 중 하나입니다.

**그래서 이 실험이 의미가 없는 걸까요? 전혀 그렇지 않습니다.** 정확한 $\alpha$ 값 자체보다 훨씬 중요한 것은 다음 두 가지 **정성적인(qualitative) 패턴**을 직접 확인했다는 점입니다.

1. 모델이 커질수록 Loss가 (대체로) 꾸준히 낮아진다 — 그리고 이 관계가 로그-로그 축에서 대략 직선을 이룬다.
2. 이 패턴은 아무 조건에서나 나타나는 것이 아니라, **① 미래를 훔쳐보지 않는 올바른 구조(Causal Mask)** 와 **② 실제로 배울 거리가 있는 데이터** 라는 두 가지 전제가 갖춰졌을 때만 나타난다.

이 두 가지를 "숫자로 외우기"가 아니라 "직접 실험해서 눈으로 확인"한 것이 이번 노트북의 핵심입니다.

## 10. 더 나아가기

이 노트북을 다 따라오셨다면, 아래와 같은 것들을 스스로 바꿔가며 실험해보는 것을 추천합니다. 코드를 조금씩 바꿔보면서 "왜 결과가 이렇게 바뀌었을까?"를 생각해보는 것이 가장 좋은 복습입니다.

- **학습 스텝 늘리기**: `n_steps`를 200 → 1000 이상으로 늘리면 어떻게 될까요? 작은 모델(XS, S)은 일찍 한계(포화)에 도달하고, 큰 모델(XL)은 계속 좋아질까요?
- **다른 시드로 반복하기**: `torch.manual_seed(0)` 부분의 숫자를 바꾸고 데이터 시드(2024)도 바꿔서 여러 번 실행해보면, 매번 alpha 값이 조금씩 다르게 나올 것입니다. 이 변동 폭 자체가 "작은 실험 하나만으로 정확한 지수를 알아내기 어렵다"는 것을 보여주는 좋은 예시입니다.
- **N_TEMPLATES(패턴의 다양성) 바꿔보기**: 300개를 30개로 줄이면(더 외우기 쉬워짐) 혹은 3000개로 늘리면(더 다양해서 어려워짐) 결과가 어떻게 달라질까요?
- **진짜 텍스트로 시도해보기**: 무작위 템플릿 대신, 아주 작은 실제 텍스트 말뭉치(예: 위키피디아 문서 일부, 소설 한 편)를 토큰화해서 사용해보면 훨씬 "언어다운" 패턴을 학습하는 모습을 볼 수 있습니다.
- **데이터 크기(D)까지 함께 바꿔보기**: 이번 실험은 모델 크기(N)만 바꿨습니다. `N_TEMPLATES`(데이터 다양성)도 함께 체계적으로 바꿔가면서, Hoffmann et al. (2022)의 $L(N, D) = E + A/N^{\alpha} + B/D^{\beta}$ 식처럼 두 변수의 효과를 동시에 관찰해볼 수도 있습니다.

수고하셨습니다! 다음 실습 코드에서 계속 이어가겠습니다.